In [193]:
from sage.all import *
import string
pretty_print_default(True)


In [194]:
x, t = var('x t')

L = lie_algebras.sl(QQ, 2, representation="matrix")

Ep, Em, H = L.gens()
Ep = Ep.matrix()
Em = Em.matrix()
H = H.matrix()

v0 = function('v0')(x, t)
A0 = v0*H

# R = LaurentPolynomialRing(QQ, 'lam')
# lam = R.gen()
lam = var('lam')
 
E = Ep + lam*Em

In [195]:
n = int(input())
fields = {}
grade = 0
count = 0

for y in range(0, int(n+1+(n+1)/2)):
    print(y % 2)
    letter = string.ascii_lowercase[y]
    exec(f"{letter} = function('{letter}')(x, t)")

    if grade not in fields:
        fields[grade] = []

    fields[grade].append(eval(letter))
    
    count += 1

    if grade % 2 == 0 and count == 1:
        grade += 1
        count = 0
    
    elif grade % 2 == 1 and count == 2:
        grade += 1
        count = 0
    
fields

0
1
0
1
0
1


{0: [a(x, t)], 1: [b(x, t), c(x, t)], 2: [d(x, t)], 3: [e(x, t), f(x, t)]}

In [196]:
def comm(A,B):
    return A*B - B*A

In [ ]:
grades = {}
fields[n][0] = 1
fields[n][1] = 1

for g in range(n, -1, -1):
    this_grade = zero_matrix(2, 2) 

    if g != 0:
        if g % 2 != 0:
            D_n = lam**((g-1)//2)*(fields[g][0]*Ep + fields[g][1]*lam*Em)
        else: 
            D_n = fields[g][0]*(lam**(g//2))*H 
        
        if (g-1) % 2 != 0:
            D_n_minus_one = lam**((g-1)//2)*(fields[g-1][0]*Ep + fields[g-1][1]*lam*Em)
        else: 
            D_n_minus_one = fields[g-1][0]*(lam**((g-1)//2))*H 
        this_grade += diff(D_n, x) 
        this_grade += comm(E, D_n_minus_one) + comm(A0, D_n) 

    else:
        this_grade -= diff(A0, t)
        D_n = fields[g][0]*(lam**(g//2))*H
        this_grade += diff(D_n, x) + comm(A0, D_n)
    grades[g] = this_grade
    # diff(D_n, x)        
grades
    


GGG 3
GGG 2
GGG 1
GGG 0


{3: [                                 0    -2*lam*d(x, t) + 2*lam*v0(x, t)]
 [2*lam^2*d(x, t) - 2*lam^2*v0(x, t)                                  0],
 2: [-lam*b(x, t) + lam*c(x, t) + lam*diff(d(x, t), x)                                                 0]
 [                                                0  lam*b(x, t) - lam*c(x, t) - lam*diff(d(x, t), x)],
 1: [                                                             0              2*b(x, t)*v0(x, t) - 2*a(x, t) + diff(b(x, t), x)]
 [-2*lam*c(x, t)*v0(x, t) + 2*lam*a(x, t) + lam*diff(c(x, t), x)                                                              0],
 0: [ diff(a(x, t), x) - diff(v0(x, t), t)                                     0]
 [                                    0 -diff(a(x, t), x) + diff(v0(x, t), t)]}

In [198]:
def split_lambda(expr):
    expr = expand(expr)
    powers = expr.collect(lam).coefficients(lam)
    out = {}
    for coeff, power in powers:
        out[power] = expand(coeff)
    return out

In [199]:
solutions = {}

fields_array = []

for kk, vv in fields.items():
    for f in vv:
        fields_array.append(f)
        solutions[f] = []


for key, value in grades.items():
    curr_field =  fields_array[key]
    print("grade: ", key)
    if key % 2 != 0:     
        print("field is: ", curr_field)   
        Eq_p = value[0,1]
        Eq_m = value[1,0]
    
        for k, v in split_lambda(Eq_p).items():
            solutions[curr_field].append(solve(v, curr_field)[0]) 
            print(k, " : ", v)

        for k, v in split_lambda(Eq_m).items():
            solutions[curr_field].append(solve(v, curr_field)) 
            print(k, " : ", v)


    else:
        print("field is: ", curr_field)    
        Eq_H = value[0,0]
        for k, v in split_lambda(Eq_H).items():
            solutions[curr_field].append(solve(v, curr_field)[0]) 
            print(k, " : ", v)
solutions = dict(reversed(solutions.items()))

print(solutions)

grade:  3
field is:  d(x, t)
1  :  -2*d(x, t) + 2*v0(x, t)
2  :  2*d(x, t) - 2*v0(x, t)
grade:  2
field is:  c(x, t)
1  :  -b(x, t) + c(x, t) + diff(d(x, t), x)
grade:  1
field is:  b(x, t)
0  :  2*b(x, t)*v0(x, t) - 2*a(x, t) + diff(b(x, t), x)
1  :  -2*c(x, t)*v0(x, t) + 2*a(x, t) + diff(c(x, t), x)
grade:  0
field is:  a(x, t)
0  :  diff(a(x, t), x) - diff(v0(x, t), t)
{1: [], d(x, t): [d(x, t) == v0(x, t), [d(x, t) == v0(x, t)]], c(x, t): [c(x, t) == b(x, t) - diff(d(x, t), x)], b(x, t): [b(x, t) == 1/2*(2*a(x, t) - diff(b(x, t), x))/v0(x, t), []], a(x, t): [diff(a(x, t), x) == diff(v0(x, t), t)]}


In [238]:
print(solutions)
equations = []
last = 0
for chave, valor in solutions.items():
   if valor:
      equations.append(valor[0])
print(equations)
position = 0
while true:
   if position == 2:
      break
   
   rules = [
      equations[position],
      diff(equations[position], x)
   ]

   print(equations[position+1].subs(rules))
   position+=1

rules = [
      equations[1],
      diff(equations[1], x)
   ]

solve(equations[2].subs(rules), a)



{1: [], d(x, t): [d(x, t) == v0(x, t), [d(x, t) == v0(x, t)]], c(x, t): [c(x, t) == b(x, t) - diff(d(x, t), x)], b(x, t): [b(x, t) == 1/2*(2*a(x, t) - diff(b(x, t), x))/v0(x, t), []], a(x, t): [diff(a(x, t), x) == diff(v0(x, t), t)]}
[d(x, t) == v0(x, t), c(x, t) == b(x, t) - diff(d(x, t), x), b(x, t) == 1/2*(2*a(x, t) - diff(b(x, t), x))/v0(x, t), diff(a(x, t), x) == diff(v0(x, t), t)]
c(x, t) == b(x, t) - diff(v0(x, t), x)
b(x, t) == 1/2*(2*a(x, t) - diff(b(x, t), x))/v0(x, t)


[a(x, t) == b(x, t)*v0(x, t) + 1/2*diff(b(x, t), x)]

In [ ]:
solutions = {}
oi = 0
for grade in sorted(grades.keys(), reverse=True):
    if oi == 0:
        oi+=1
        continue 
    print("\nGRADE:", grade)

    eq_matrix = grades[grade].subs(solutions)

    # =====================================
    # ODD GRADES
    # =====================================

    if grade % 2 != 0:

        Eq_p = expand(eq_matrix[0,1])
        Eq_m = expand(eq_matrix[1,0])

        print("Ep equation:")
        show(Eq_p)

        print("Em equation:")
        show(Eq_m)

        # ---------------------------------
        # solve Ep projection first
        # ---------------------------------

        split_p = split_lambda(Eq_p)

        for power, expr in split_p.items():

            expr = expand(expr)

            print("lambda^", power, ":", expr)

            unknown = fields[grade][0]

            sol = solve(expr, unknown)

            if sol:

                solutions[unknown] = sol[0].rhs()

                print("Solved:")
                show(unknown == solutions[unknown])

        # ---------------------------------
        # substitute immediately
        # ---------------------------------

        eq_matrix = eq_matrix.subs(solutions)

        # ---------------------------------
        # now Em projection
        # ---------------------------------

        Eq_m = expand(eq_matrix[1,0])

        split_m = split_lambda(Eq_m)

        for power, expr in split_m.items():

            expr = expand(expr)

            print("lambda^", power, ":", expr)

            unknown = fields[grade][1]

            sol = solve(expr, unknown)

            if sol:

                solutions[unknown] = sol[0].rhs()

                print("Solved:")
                show(unknown == solutions[unknown])

    # =====================================
    # EVEN GRADES
    # =====================================

    else:

        Eq_H = expand(eq_matrix[0,0])

        print("H equation:")
        show(Eq_H)

        split_h = split_lambda(Eq_H)

        for power, expr in split_h.items():

            expr = expand(expr)

            print("lambda^", power, ":", expr)

            unknown = fields[grade][0]

            sol = solve(expr, unknown)

            if sol:

                solutions[unknown] = sol[0].rhs()

                print("Solved:")
                show(unknown == solutions[unknown])

print("\nFINAL SOLUTIONS")

for k,v in solutions.items():
    show(k == v)

# solutions = {}

# fields_array = [v]

# for kk, vv in fields.items():
#     for f in vv:
#         fields_array.append(f)

# for key, value in grades.items():

#     eq_matrix = grades[grade].subs(solutions)

#     print("grade: ", key)
#     if key % 2 != 0:     
#         print("field is: ", fields_array[key])   
#         Eq_p = value[0,1]
#         Eq_m = value[1,0]
#         Eq_E = Eq_p + Eq_m

#         for k, v in split_lambda(Eq_E).items():
            
#             print(k, " : ", v)
#             solutions[fields_array[key]] = solve(v, fields_array[key])
#             print("CC: ", v, fields_array[key])

#     else:
#         print("field is: ", fields_array[key])    
#         Eq_H = value[0,0]
#         for k, v in split_lambda(Eq_H).items():
           
#             print(k, " : ", v)
#             solutions[fields_array[key]] = solve(v, fields_array[key])

# print(solutions)
        


GRADE: 2
H equation:
-lam*b(x, t) + lam*c(x, t) + lam*diff(d(x, t), x)
lambda^ 1 : -b(x, t) + c(x, t) + diff(d(x, t), x)
Solved:
d(x, t) == b(x, t) - c(x, t)

GRADE: 1
Ep equation:
2*b(x, t)*v(x, t) - 2*a(x, t) + diff(b(x, t), x)
Em equation:
-2*lam*c(x, t)*v(x, t) + 2*lam*a(x, t) + lam*diff(c(x, t), x)
lambda^ 0 : 2*b(x, t)*v(x, t) - 2*a(x, t) + diff(b(x, t), x)
Solved:
b(x, t) == 1/2*(2*a(x, t) - diff(b(x, t), x))/v(x, t)
lambda^ 1 : -2*c(x, t)*v(x, t) + 2*a(x, t) + diff(c(x, t), x)
Solved:
c(x, t) == 1/2*(2*a(x, t) + diff(c(x, t), x))/v(x, t)

GRADE: 0
H equation:
diff(a(x, t), x) - diff(v(x, t), t)
lambda^ 0 : diff(a(x, t), x) - diff(v(x, t), t)
Solved:
a(x, t) == diff(v(x, t), t)

FINAL SOLUTIONS
d(x, t) == b(x, t) - c(x, t)
b(x, t) == 1/2*(2*a(x, t) - diff(b(x, t), x))/v(x, t)
c(x, t) == 1/2*(2*a(x, t) + diff(c(x, t), x))/v(x, t)
a(x, t) == diff(v(x, t), t)


In [ ]:
fields

{0: [a(x, t)], 1: [b(x, t), c(x, t)], 2: [d(x, t)], 3: [1, 1]}